In [1]:
import pandas as pd
import numpy as np
import json
import math
import os

# =========================
# 0) CONFIG
# =========================

PATHS = {
    "PPB":  "../../data/PPB_Affinity/PPB-Affinity.xlsx", 
    "PPB_AF": "../../data/PPB_Affinity/PPB-Affinity-AF.xlsx", 
    "PPB_AF_PLDDT": "../extracted_plddt_scores.csv",
    "OVERATH": "../../data/Overath/final_dataset.csv",
    "EGFR_R1": "../../data/Adaptyv/result_summary_EGFR_round1.csv",
    "EGFR_R2": "../../data/Adaptyv/result_summary_EGFR_round2.csv",
    "NIPAH": "../../data/Adaptyv/results_nipah.csv",
}

# KD threshold for deriving a binder/non-binder label when KD exists
KD_BINDER_THRESHOLD_M = 1e-5  # 10 µM = 10,000 nM (Overath threshold)


# =========================
# 1) HELPERS
# =========================

def compute_pKD(kd):
    """pKD = -log10(KD) where KD is in molar."""
    try:
        kd = float(kd)
    except Exception:
        return np.nan
    if not np.isfinite(kd) or kd <= 0:
        return np.nan
    return -math.log10(kd)

def parse_evaluations(cell):
    """Nipah evaluations are stored as JSON strings (list of dicts)."""
    if pd.isna(cell):
        return []
    if isinstance(cell, list):
        return cell
    try:
        return json.loads(cell)
    except Exception:
        return []

def extract_eval_value(evals, metric, target="nipah-glycoprotein-g", target_required=True):
    """
    Extract the first matching metric value from a list of evaluation dicts.
    If target_required=True, match dict['target']==target.
    """
    for e in evals:
        if not isinstance(e, dict):
            continue
        if e.get("metric") != metric:
            continue
        if target_required and e.get("target") != target:
            continue
        return e.get("value")
    return np.nan

def infer_assay_method_nipah(evals, target="nipah-glycoprotein-g"):
    """
    Coarse inference: if curve data exists -> SPR/BLI; else unknown.
    """
    spr = extract_eval_value(evals, "spr_kinetic_curves", target=target, target_required=True)
    bli = extract_eval_value(evals, "bli_kinetic_curves", target=target, target_required=True)

    if isinstance(spr, (dict, list)) and len(spr) != 0:
        return "SPR"
    if isinstance(bli, (dict, list)) and len(bli) != 0:
        return "BLI"

    # if kinetic/affinity values exist but no curves
    kd = extract_eval_value(evals, "kd", target=target, target_required=True)
    kon = extract_eval_value(evals, "kon", target=target, target_required=True)
    koff = extract_eval_value(evals, "koff", target=target, target_required=True)
    if pd.notna(kd) or pd.notna(kon) or pd.notna(koff):
        return "kinetics_unknown"

    return np.nan

def join_nonempty(*vals, sep=" | "):
    cleaned = []
    for v in vals:
        if v is None or (isinstance(v, float) and np.isnan(v)):
            continue
        s = str(v).strip()
        if s == "" or s.lower() in ["nan", "none"]:
            continue
        cleaned.append(s)
    return sep.join(cleaned) if cleaned else np.nan

def to_int_binary(series: pd.Series) -> pd.Series:
    # bool -> 0/1
    if pd.api.types.is_bool_dtype(series) or str(series.dtype) == "boolean":
        return series.astype("Int64")

    # numeric -> 0/1 nur wenn wirklich 0/1
    if pd.api.types.is_numeric_dtype(series):
        s = series.copy()
        # alles außer 0/1 wird NA
        s = s.where(s.isin([0, 1]), pd.NA)
        return s.astype("Int64")

    # strings -> map
    s = series.astype(str).str.strip().str.lower()
    mapped = s.map({
        "true": 1, "false": 0,
        "1": 1, "0": 0,
        "1.0": 1, "0.0": 0
    })
    return mapped.astype("Int64")



# =========================
# 2) STANDARDIZERS (DIRECT MAPPING)
# =========================
# Canonical columns created in all standardized dataframes:
CANON = [
    "dataset", "source_dataset",
    "target",
    "binder", "sequence",
    "kd_M", "pKD",
    "binding_binary", "binding_strength",
    "assay_method", "structure_method", "design_method",
    "plddt", "pae_interaction", "iptm", "ipSAE_min", "esm_pll",
    "rosetta_interface_dG"
]

def init_out(df):
    return pd.DataFrame(index=df.index, columns=CANON)

def standardize_ppb(df, dataset="PPB-Affinity"):
    out = init_out(df)
    out["dataset"] = dataset
    out["source_dataset"] = df.get("Source Data Set")

    # target/binder naming
    out["target"] = df.get("Receptor Name")
    out["binder"] = df.get("Ligand Name")
    out["sequence"] = np.nan  # PPB does not include sequences

    # experimental affinity
    out["kd_M"] = pd.to_numeric(df.get("KD(M)"), errors="coerce")
    out["pKD"] = out["kd_M"].apply(compute_pKD)

    # Only derive binary label where KD exists (don’t treat missing KD as non-binder)
    out["binding_binary"] = np.where(
        out["kd_M"].notna(),
        out["kd_M"] < KD_BINDER_THRESHOLD_M,
        np.nan
    )

    out["assay_method"] = df.get("Affinity Method")
    out["structure_method"] = df.get("Structure Method")  # often experimental (X-RAY, NMR)
    out["design_method"] = np.nan

    # no in-silico metrics here
    return out.reset_index(drop=True)

def standardize_ppb_af(df, plddt_scores,dataset="PPB-Affinity-AF"):
    out = standardize_ppb(df, dataset=dataset)
    # optional extra field: binding free energy if present
    if "dG(kcal/mol)" in df.columns:
        out["dG_kcal_mol"] = pd.to_numeric(df.get("dG(kcal/mol)"), errors="coerce")
    out["plddt"] = df['PDB'].map(plddt_scores.set_index('pdb_id')['extracted_plddt']
)
    return out

def standardize_overath(df):
    out = init_out(df)

    # Drop duplicates
    df = df.drop(df[df['source'] == "Adaptyv binder comp round 1"].index)
    df = df.drop(df[df['source'] == "Adaptyv binder comp round 2"].index)

    out["dataset"] = "Overath"
    out["source_dataset"] = df.get("source")

    out["target"] = df.get("target_id")
    out["binder"] = df.get("binder_id")

    # sequence: Overath uses A_seq
    out["sequence"] = df.get("A_seq")

    # Overath provides binary experimental success label, no KD
    out["binding_binary"] = df.get("binder")

    out["structure_method"] = "predicted_model"
    out["assay_method"] = np.nan
    out["design_method"] = np.nan

    # computational metrics (key ones)
    out["plddt"] = df.get("af2_plddt_binder")
    out["pae_interaction"] = df.get("af2_pae_interaction")
    out["iptm"] = df.get("af3_iptm_avg")
    out["ipSAE_min"] = df.get("af3_ipSAE_min")
    out["rosetta_interface_dG"] = df.get("af3_rosetta_interface_dG")

    return out.reset_index(drop=True)

def standardize_egfr_r1(df):
    out = init_out(df)
    out["dataset"] = "Adaptyv_EGFR_R1"
    out["source_dataset"] = "Adaptyv EGFR Binder Challenge - Round 1"
    out["target"] = "EGFR"

    out["binder"] = df["sequence_name"]
    out["sequence"] = df.get("sequence")

    out["kd_M"] = pd.to_numeric(df.get("kd"), errors="coerce")
    out["pKD"] = out["kd_M"].apply(compute_pKD)

    # Only derive binary label where KD exists (don’t treat missing KD as non-binder)
    out["binding_binary"] = np.where(
        out["kd_M"].notna(),
        out["kd_M"] < KD_BINDER_THRESHOLD_M,
        np.nan
    )

    out["structure_method"] = "predicted_model"
    out["assay_method"] = np.nan  
    out["design_method"] = df.apply(lambda r: join_nonempty(r.get("model_names"), r.get("methods")), axis=1)

    out["plddt"] = pd.to_numeric(df.get("plddt"), errors="coerce")
    out["pae_interaction"] = pd.to_numeric(df.get("pae_interaction"), errors="coerce")

    return out.reset_index(drop=True)

def standardize_egfr_r2(df):
    out = init_out(df)
    out["dataset"] = "Adaptyv_EGFR_R2"
    out["source_dataset"] = "Adaptyv EGFR Binder Challenge - Round 2"
    out["target"] = "EGFR"

    out["binder"] = df.get("name")
    out["sequence"] = df.get("sequence")

    out["kd_M"] = pd.to_numeric(df.get("kd"), errors="coerce")
    out["pKD"] = out["kd_M"].apply(compute_pKD)

    # binary label is given in round2
    out["binding_binary"] = df.get("binding")
    out["binding_strength"] = df.get("binding_strength")

    out["structure_method"] = "predicted_model"
    out["assay_method"] = np.nan
    out["design_method"] = df.get("design_models")

    out["plddt"] = pd.to_numeric(df.get("plddt"), errors="coerce")
    out["pae_interaction"] = pd.to_numeric(df.get("pae_interaction"), errors="coerce")
    out["iptm"] = pd.to_numeric(df.get("iptm"), errors="coerce")
    out["esm_pll"] = pd.to_numeric(df.get("esm_pll"), errors="coerce")

    return out.reset_index(drop=True)

def standardize_nipah(df):
    out = init_out(df)
    out["dataset"] = "Adaptyv_NIPAH"
    out["source_dataset"] = "Nipah Binder Competition Results"
    out["target"] = "nipah-glycoprotein-g"

    out["binder"] = df.get("name")
    out["sequence"] = df.get("sequence")
    out["design_method"] = df.get("designMethod")
    out["structure_method"] = "predicted_model"

    evals = df["evaluations"].apply(parse_evaluations)

    # experimental signals / affinity
    out["binding_binary"] = evals.apply(lambda ev: extract_eval_value(ev, "binding", target="nipah-glycoprotein-g", target_required=True))
    out["binding_strength"] = evals.apply(lambda ev: extract_eval_value(ev, "binding_strength", target="nipah-glycoprotein-g", target_required=True))

    out["kd_M"] = pd.to_numeric(
        evals.apply(lambda ev: extract_eval_value(ev, "kd", target="nipah-glycoprotein-g", target_required=True)),
        errors="coerce"
    )
    out["pKD"] = out["kd_M"].apply(compute_pKD)

    out["assay_method"] = evals.apply(lambda ev: infer_assay_method_nipah(ev, target="nipah-glycoprotein-g"))

    # computational metrics (Boltz2 / ESMFold etc.)
    # pLDDT: prefer ESMFold if available, otherwise Boltz2
    plddt_esm = pd.to_numeric(evals.apply(lambda ev: extract_eval_value(ev, "esmfold_plddt", target=None, target_required=False)), errors="coerce")
    plddt_boltz = pd.to_numeric(evals.apply(lambda ev: extract_eval_value(ev, "boltz2_plddt", target=None, target_required=False)), errors="coerce")
    out["plddt"] = plddt_esm.where(plddt_esm.notna(), plddt_boltz)

    out["iptm"] = pd.to_numeric(evals.apply(lambda ev: extract_eval_value(ev, "boltz2_iptm", target=None, target_required=False)), errors="coerce")
    out["ipSAE_min"] = pd.to_numeric(evals.apply(lambda ev: extract_eval_value(ev, "boltz2_min_ipsae", target=None, target_required=False)), errors="coerce")

    # NOTE: Nipah export does not provide a scalar PAE interaction; keep NA by default.
    # If you want a proxy, you could map out["pae_interaction"] to boltz2_complex_pde:
    # out["pae_interaction"] = pd.to_numeric(evals.apply(lambda ev: extract_eval_value(ev, "boltz2_complex_pde", target=None, target_required=False)), errors="coerce")

    return out.reset_index(drop=True)


# =========================
# 3) LOAD RAW DATA
# =========================

ppb_raw = pd.read_excel(PATHS["PPB"])
ppb_af_raw = pd.read_excel(PATHS["PPB_AF"])
ppb_af_plddt_raw = pd.read_csv(PATHS["PPB_AF_PLDDT"])
over_raw = pd.read_csv(PATHS["OVERATH"])
egfr1_raw = pd.read_csv(PATHS["EGFR_R1"])
egfr2_raw = pd.read_csv(PATHS["EGFR_R2"])
nipah_raw = pd.read_csv(PATHS["NIPAH"])


# =========================
# 4) STANDARDIZE + MERGE
# =========================

ppb_std = standardize_ppb(ppb_raw)
ppb_af_std = standardize_ppb_af(ppb_af_raw, ppb_af_plddt_raw)
over_std = standardize_overath(over_raw)
egfr1_std = standardize_egfr_r1(egfr1_raw)
egfr2_std = standardize_egfr_r2(egfr2_raw)
nipah_std = standardize_nipah(nipah_raw)

dfs = [ppb_std, ppb_af_std, over_std, egfr1_std, egfr2_std, nipah_std]
merged = pd.concat(dfs, ignore_index=True, sort=False)

merged["binding_binary"] = to_int_binary(merged["binding_binary"])

# =========================
# 5) SANITY CHECKS + EXPORT
# =========================

print("Merged shape:", merged.shape)
print("\nRows per dataset:")
print(merged["dataset"].value_counts(dropna=False))

print("\nKD availability (fraction of rows with KD) per dataset:")
print(merged.assign(has_kd=merged["kd_M"].notna()).groupby("dataset")["has_kd"].mean().sort_values(ascending=False))

print(merged.head())

# export
merged.to_csv("merged_all.csv", index=False)
print("\nWrote: merged_all.csv")


C:\Users\Emma Lux\AppData\Local\Temp\ipykernel_1236\589981587.py:328: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  merged = pd.concat(dfs, ignore_index=True, sort=False)


Merged shape: (17467, 19)

Rows per dataset:
dataset
PPB-Affinity       12062
Overath             3676
Adaptyv_NIPAH       1030
Adaptyv_EGFR_R2      402
Adaptyv_EGFR_R1      202
PPB-Affinity-AF       95
Name: count, dtype: int64

KD availability (fraction of rows with KD) per dataset:
dataset
PPB-Affinity       1.000000
PPB-Affinity-AF    1.000000
Adaptyv_EGFR_R2    0.136816
Adaptyv_NIPAH      0.099029
Adaptyv_EGFR_R1    0.039604
Overath            0.000000
Name: has_kd, dtype: float64
        dataset source_dataset                target  \
0  PPB-Affinity    SKEMPI v2.0   hGH binding protein   
1  PPB-Affinity    SKEMPI v2.0            Angiogenin   
2  PPB-Affinity    SKEMPI v2.0               Eglin c   
3  PPB-Affinity    SKEMPI v2.0         Tissue factor   
4  PPB-Affinity    SKEMPI v2.0  HIV-1 capsid protein   

                      binder sequence          kd_M        pKD  \
0       Human growth hormone      NaN  9.000000e-10   9.045757   
1     Ribonuclease inhibitor      NaN  5